# Diagnostic — does raw (non-normalized) distance find movement cosine drift hides? (colab_11 v2)

colab_11 v2's detector #1 came back INERT (held-out drift 0.0051) despite a genuinely better-fitting adapter than v1 -- a real, widening loss improvement at every checkpoint. Cosine distance is scale-invariant by construction: it measures only the angle between two vectors, never their length. If continued pretraining shifts the pooled embedding's magnitude without rotating it much, cosine distance stays near-zero while the raw distance between the same two vectors would not. This is a standalone, read-only diagnostic -- it loads the already-saved zero-shot and CPT-v2 embeddings from Drive and recomputes distance metrics on them; it re-runs no part of the training or embedding pipeline and needs no GPU.

### Setup — install `anndata`

Colab's base runtime ships `numpy`/`pandas` but not `anndata`, needed to read the saved `.h5ad` embeddings. `anndata` is **pinned to `0.12.19`** — the exact version that wrote these files (colab_09/colab_11 env). Pinning to that older version matters: an unpinned `pip install anndata` pulls the newest release, which drags `numpy`/`pandas` up to bleeding-edge versions on top of Colab's base and breaks `numpy` with a load-time version skew. Because `0.12.19`'s requirements are already satisfied by Colab's base `numpy`/`pandas`, pip leaves them untouched. The resolved version prints below for the run's software-version record.

In [ ]:
!pip install -q "anndata==0.12.19"
import anndata
print("anndata", anndata.__version__)

### 1a — load both saved embeddings, align by `cell_index`, reproduce the known cosine drift

Load the frozen zero-shot baseline (`glia_geneformer_zeroshot.h5ad`, colab_09) and the v2 post-CPT embedding (`glia_geneformer_cpt_aggregated_v2_seed0.h5ad`, colab_11), align them cell-for-cell by `cell_index` exactly as colab_11 §7a did, and recompute the same per-cell cosine drift metric. Reproducing 0.0050/0.0051 here before computing anything new confirms the alignment logic is correct -- if this diagnostic's own cosine number doesn't match colab_11's, nothing built on top of it can be trusted.

In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
CPT_V2_PATH   = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_cpt_aggregated_v2_seed0.h5ad")
for p in (ZEROSHOT_PATH, CPT_V2_PATH):
    assert os.path.exists(p), f"missing saved embedding: {p}"

zs  = ad.read_h5ad(ZEROSHOT_PATH)
cpt = ad.read_h5ad(CPT_V2_PATH)
print("zero-shot:", zs.shape, "| cpt v2:", cpt.shape)

# align by cell_index exactly as colab_11 §7a did (cpt carries the authoritative train/test split)
zs_df  = pd.DataFrame(zs.X,  index=zs.obs["cell_index"].values)
cpt_df = pd.DataFrame(cpt.X, index=cpt.obs["cell_index"].values)
common = cpt.obs["cell_index"].values
zs_aligned = zs_df.reindex(common)
assert zs_aligned.notna().all().all(), "zero-shot rows missing after cell_index alignment"

A = zs_aligned.to_numpy(dtype=np.float32)   # zero-shot
B = cpt_df.to_numpy(dtype=np.float32)       # post-CPT v2
assert A.shape == B.shape
split = cpt.obs["split"].values
test_mask = split == "test"
assert test_mask.any()

def _per_cell_cosine(X, Y):
    num = (X * Y).sum(1)
    den = np.linalg.norm(X, axis=1) * np.linalg.norm(Y, axis=1) + 1e-12
    return num / den

cos = _per_cell_cosine(A, B)
cos_drift_all  = 1.0 - float(np.median(cos))
cos_drift_test = 1.0 - float(np.median(cos[test_mask]))
print(f"reproduced cosine drift -- all: {cos_drift_all:.4f} | test: {cos_drift_test:.4f}")
print("colab_11 v2 §7a recorded  -- all: 0.0050 | test: 0.0051")


### 2a — do the two embeddings sit at different typical lengths?

Cosine distance can't see a pure magnitude shift by construction. Before testing raw distance, check the more basic question directly: does the post-CPT embedding's typical vector length (its norm) differ from the zero-shot one's? Report the median norm of each, split by held-out test vs train (the gate's actual surface) and all cells for context, plus the ratio between them -- a ratio far from 1 would mean CPT systematically stretches or shrinks the pooled vector.

In [ ]:
normA = np.linalg.norm(A, axis=1)
normB = np.linalg.norm(B, axis=1)

for name, mask in [("all", np.ones(len(A), dtype=bool)), ("train", split == "train"), ("test", test_mask)]:
    mA, mB = float(np.median(normA[mask])), float(np.median(normB[mask]))
    print(f"{name:5s} | median ||zero-shot||: {mA:8.3f} | median ||cpt-v2||: {mB:8.3f} | ratio: {mB/mA:.4f}")


### 3a — raw (non-normalized) per-cell distance, on the same scale as cosine distance

Compute the raw Euclidean distance between each cell's zero-shot and post-CPT-v2 vector, then normalize it by the pair's own typical length (`||A-B|| / mean(||A||, ||B||)`) so it lands on a comparable dimensionless scale to cosine distance instead of an arbitrary raw number. If this relative distance is also ~0.005 on the held-out test split, magnitude movement is just as flat as direction and this diagnostic is a null -- the mechanism is more likely mean-pooling than cosine-normalization. If it's meaningfully larger than the cosine drift, a real magnitude-driven shift was there all along, one the angle-only metric structurally could not see.

In [ ]:
raw_dist = np.linalg.norm(A - B, axis=1)
typical_len = (normA + normB) / 2.0
rel_dist = raw_dist / typical_len

for name, mask in [("all", np.ones(len(A), dtype=bool)), ("train", split == "train"), ("test", test_mask)]:
    print(f"{name:5s} | median raw ||A-B||: {np.median(raw_dist[mask]):8.4f} "
          f"| median relative dist: {np.median(rel_dist[mask]):.4f}")

print(f"\nfor reference -- cosine drift (test): {cos_drift_test:.4f}")
